In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
dbutils.widgets.text("incremental_run","0")


In [0]:
incremental_load = dbutils.widgets.get("incremental_run")
print(incremental_load)

In [0]:
df = spark.sql('''
               SELECT * FROM parquet.`abfss://silver@storageforcarsproject.dfs.core.windows.net/OBT/`''')

display(df)

In [0]:
df_src = spark.sql('''
                   SELECT DISTINCT(Dealer_ID),DealerName FROM PARQUET.`abfss://silver@storageforcarsproject.dfs.core.windows.net/OBT/`''')
display(df_src)

In [0]:
if spark.catalog.tableExists("cars_cata.gold.dim_dealer"):
    df_sink = spark.sql('''
                        SELECT dim_dealer_key,Dealer_ID,DealerName FROM cars_cata.gold.dim_dealer''')
else:
    df_sink = spark.sql('''
                        SELECT 1 as dim_dealer_key,Dealer_id,DealerName FROM parquet.`abfss://silver@storageforcarsproject.dfs.core.windows.net/OBT/` WHERE 1=0''')

In [0]:
df_total = df_src.join(df_sink,df_src["Dealer_ID"]==df_sink["Dealer_ID"],"LEFT").select(df_src["Dealer_ID"],df_src["DealerName"],df_sink["dim_dealer_key"])

### # New Record

In [0]:
df_new = df_total.filter(col("dim_dealer_key").isNull()).select(df_src["Dealer_ID"],df_src["DealerName"])

### Old Record

In [0]:
df_old = df_total.filter(col("dim_dealer_key").isNotNull())

### ## surrogate key for new record

In [0]:
if incremental_load == '0':
    max_value = 1
else:
    max_value_df = spark.sql('''
                             SELECT max(dim_dealer_key) FROM cars_cata.gold.dim_dealer''')
    max_value = max_value_df.collect()[0][0] + 1


In [0]:
df_new = df_new.withColumn('dim_dealer_key',max_value + monotonically_increasing_id())


In [0]:
df_final = df_new.union(df_old)


## SCD TYPE1

In [0]:
from delta.tables import DeltaTable

In [0]:
if spark.catalog.tableExists("cars_cata.gold.dim_dealer"):
    delta_obj = DeltaTable.forPath(spark,"abfss://gold@storageforcarsproject.dfs.core.windows.net/dim_dealer")
    delta_obj.alias("trg").merge(df_final.alias("src"),"trg.Dealer_ID == src.Dealer_ID")\
        .whenMatchedUpdateAll()\
        .whenNotMatchedInsertAll()\
        .execute()
else:
    df_final.write.format("delta")\
        .mode("overwrite")\
        .option("path","abfss://gold@storageforcarsproject.dfs.core.windows.net/dim_dealer")\
        .saveAsTable("cars_cata.gold.dim_dealer")
        

In [0]:
%sql
select * FROM delta.`abfss://gold@storageforcarsproject.dfs.core.windows.net/dim_dealer`